# AI651 PA2 - Question 1 diagram and Question 2 (Motion or Memory?)  [version 2a]

Clones the public repo (branch `PA2`) and runs everything top to bottom: **Runtime -> Change runtime type -> T4 GPU, then Run all** (about 1 hour; each model trains in about 5 minutes). Push your latest commit first, because the notebook clones the repo. Nothing uses Google Drive.

In [ ]:
# 1. Setup
import os, torch
REPO = 'https://github.com/RazaTypesCode/DLSTG.git'
BRANCH = 'PA2'
if not os.path.exists('/content/DLSTG'):
    !git clone --branch $BRANCH --single-branch $REPO /content/DLSTG
else:
    !git -C /content/DLSTG pull
os.environ['PA2_RUNS'] = '/content/runs'
os.environ['PA2_DATA'] = '/content/data'
os.makedirs('/content/runs', exist_ok=True)
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - switch runtime to T4 GPU')
!git -C /content/DLSTG log --oneline -3

In [ ]:
# 2. Question 1: architecture diagram
%cd /content/DLSTG/Question1
!python q1_diagram.py
from IPython.display import Image, display
display(Image('figures/q1_architecture.png'))
%cd /content/DLSTG/Question2

In [ ]:
# 3. Question 2: Table 1 (transform and window checks) and Figure 1 (optical flow)
%cd /content/DLSTG/Question2
!python check_table1.py
!python fig1_flow.py
from IPython.display import Image, display
display(Image('figures/fig1.png'))

In [ ]:
# 4. Model A (seeds 0, 1), then Table 2 and Figure 2
for s in (0, 1):
    !python train.py --model A --seed $s
!python stage_A1.py
!cat results/table2.md
display(Image('figures/fig2.png'))

In [ ]:
# 5. Stress tests on model A: Tables 3-4, Figures 3-5
!python stage_A2.py
!cat results/table3.md results/table4.md
import glob
for f in sorted(glob.glob('figures/fig3_*.png')) + ['figures/fig4.png', 'figures/fig5.png']:
    print(f); display(Image(f))

In [ ]:
# 6. Models B, C and the two Model D candidates (seed 0)
for m, s in [('B', 0), ('B', 1), ('C', 0), ('C', 1), ('D_p8', 0), ('D_dil2', 0)]:
    !python train.py --model $m --seed $s

In [ ]:
# 7. Choose Model D on validation only, train its seed 1, then Tables 5-6 and Figure 6
!python select_D.py
!python train.py --model D --seed 1
!python stage_final.py
!cat results/table5.md results/table6.md
for f in sorted(glob.glob('figures/fig6a_*.png')) + ['figures/fig6b.png']:
    print(f); display(Image(f))

In [ ]:
# 8. Timing record and download of all outputs (tables, figures, GIFs)
import json
for f in sorted(glob.glob('/content/runs/*/log.json')):
    d = json.load(open(f))
    print(f"{d['model']:7s} seed {d['seed']}  params {d['params']:>8,}  reach {d['reach_px']:>2} px  {d['mean_epoch_sec']:.1f} s/epoch  [{d['hardware']}]")
%cd /content/DLSTG
!zip -qr /content/pa2_outputs.zip Question1/figures Question2/results Question2/figures
from google.colab import files
files.download('/content/pa2_outputs.zip')